# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All dataset resources (record sets, fields, columns) are referenced by their `@id`, following the Croissant schema.

### Dataset Source
The data is described by a Croissant schema at:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and explore its structure.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset and show metadata
dataset = mlc.Dataset(croissant_url)

print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")


## 2. Data Overview
Examine the structure of the dataset, including record sets, fields, and `@id` values.

In [ ]:
# List all record sets and their fields, referencing by `@id`
print("Available record sets and their fields (referenced by @id):\n")

for record_set in dataset.metadata.record_sets:
    print(f"Record Set Name: {record_set.name}")
    print(f"    @id: {record_set.id}")
    print("    Fields:")
    for field in record_set.fields:
        print(f"        - Field Name: {field.name} | @id: {field.id}")
    print()

## 3. Data Extraction
We will load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Gather all record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print(f"Record set IDs found: {record_set_ids}\n")

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# For demonstration, select the first record set
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"Fields for '{main_record_set_id}': {dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets available in the dataset.")


## 4. Exploratory Data Analysis (EDA)
Let's perform common data processing steps (filtering, normalization, grouping) using field `@id`s.

- **Select a numeric field**: We'll try to auto-detect a numeric field; alternatively, set one from above. 
- **Group by**: Also select a categorical field.

In [ ]:
import numpy as np

if main_record_set_id:
    df = dataframes[main_record_set_id].copy()

    # Attempt to auto-detect a numeric field by looking for float/int columns
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Using numeric field: {numeric_field_id}")
    else:
        print("No numeric fields found. Please manually set 'numeric_field_id'.")
        numeric_field_id = None

    # Attempt to auto-detect a categorical/grouping field
    group_candidates = df.select_dtypes(include=['object', 'category']).columns.tolist()
    group_field_id = group_candidates[0] if group_candidates else None

    threshold = df[numeric_field_id].mean() if numeric_field_id else None
    if numeric_field_id and threshold is not None:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by categorical field and compute means
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id} (showing means):")
            display(grouped_df.head())
        else:
            print("No grouping field found.")
    else:
        print("Numeric field or threshold not found for EDA.")
else:
    print("No main record set available for EDA.")


## 5. Visualization
Visualize numeric field distribution and correlation with another field (if present).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id:
    # Distribution plot
    plt.figure(figsize=(7,4))
    sns.histplot(dataframes[main_record_set_id][numeric_field_id], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Correlation scatterplot if there are at least two numeric fields
    if len(numeric_candidates) >= 2:
        other_field = numeric_candidates[1]
        plt.figure(figsize=(6,6))
        sns.scatterplot(x=numeric_field_id, y=other_field, data=dataframes[main_record_set_id])
        plt.title(f"{numeric_field_id} vs {other_field}")
        plt.show()
else:
    print("Not enough numeric fields available for visualization.")


## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and analyze a Croissant-described dataset with the `mlcroissant` library, referencing all entities by their `@id`.

**Key steps:**
- Loaded dataset schema and metadata
- Identified record sets, fields, and referenced them via `@id`
- Extracted tabular data into pandas DataFrames
- Conducted simple EDA and visualization via numeric and categorical fields

For further work, you could apply more advanced machine learning, statistical analysis, or data integration using the field `@id` system for reliable programmatic exploration.